**Prelude:** This notebook walks you through the most important PySpark APIs, from connecting to a Spark cluster all the way to streaming and machine learning. This tutorial is based off the documentation of PySpark (as of version 4.2.0).

### 1. Initialization ###

In [1]:
import os, sys, socket, time, shutil

# Ensure PySpark workers use the correct Python executable on Windows
venv_python = os.path.join(sys.prefix, 'Scripts', 'python.exe')
os.environ['PYSPARK_PYTHON'] = venv_python
os.environ['PYSPARK_DRIVER_PYTHON'] = venv_python

# Required for Hadoop local file operations on Windows (winutils.exe)
os.environ['HADOOP_HOME'] = r'C:\hadoop'

import pyspark
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.functions import col, sum, avg, window, from_json, randn

from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

import pandas as pd
import pyspark.pandas as ps
import numpy as np

print(f"PySpark version: {pyspark.__version__}")
print(f"Python executable: {sys.executable}")
print(f"PYSPARK_PYTHON: {os.environ.get('PYSPARK_PYTHON', 'NOT SET')}")
print(f"PYSPARK_DRIVER_PYTHON: {os.environ.get('PYSPARK_DRIVER_PYTHON', 'NOT SET')}")
print(f"HADOOP_HOME: {os.environ.get('HADOOP_HOME', 'NOT SET')}")


c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Lib\site-packages\pyspark\pandas\__init__.py:33: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


PySpark version: 4.2.0
Python executable: c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Scripts\python.exe
PYSPARK_PYTHON: c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Scripts\python.exe
PYSPARK_DRIVER_PYTHON: c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Scripts\python.exe
HADOOP_HOME: C:\hadoop


### 2. Python Spark Connect Client ###

Spark Connect is a client-server architecture that lets you connect to a remote Spark cluster from any Python environment. Below is the `SparkSession` creation that connects to a local Spark Connect Server (replace the host/port if using a remote cluster).

In [2]:
import os, sys, socket

venv_python = os.path.join(sys.prefix, 'Scripts', 'python.exe')
print('Expected venv Python: ' + venv_python)
print(f'sys.executable: {sys.executable}')
print(f'Hostname: {socket.gethostname()}')
try:
    print(f'Hostname resolves to: {socket.gethostbyname(socket.gethostname())}')
except Exception as e:
    print(f'Hostname resolution failed: {e}')

# Force driver to bind on 127.0.0.1 to avoid Windows hostname/IPv6 bind issues
driver_host = '127.0.0.1'
print(f'Setting spark.driver.host = {driver_host}')

spark = (
    SparkSession.builder
    .appName('PySpark Tutorial')
    .master('local[*]')
    .config('spark.driver.host', driver_host)
    .config('spark.driver.bindAddress', driver_host)
    .config('spark.pyspark.python', venv_python)
    .config('spark.pyspark.driver.python', venv_python)
    .config('spark.executor.memory', '512m')
    .config('spark.hadoop.fs.permissions.umask-mode', '000')
    .getOrCreate()
)

print('Spark session created successfully')
print(f'Spark version: {spark.version}')
print(f'Spark UI: http://{spark.conf.get("spark.driver.host")}:4040')


Expected venv Python: c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Scripts\python.exe
sys.executable: c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Scripts\python.exe
Hostname: DESKTOP-VE60JNT
Hostname resolves to: 192.168.1.43
Setting spark.driver.host = 127.0.0.1


c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark session created successfully
Spark version: 4.2.0
Spark UI: http://127.0.0.1:4040


### 3. Spark SQL & DataFrames ###

DataFrames are the core data structure for structured data processing. They allow you to mix Python and SQL seamlessly. 

#### 3.1. Create a sample DataFrame ####

In [3]:
# We’ll create a small sales dataset and register it as a temporary view.

# Sample sales data
data = [
    ("2025-01-01", "ProductA", 120, 10),
    ("2025-01-02", "ProductB", 200, 5),
    ("2025-01-02", "ProductA", 150, 8),
    ("2025-01-03", "ProductC", 300, 3),
    ("2025-01-03", "ProductB", 250, 6),
    ("2025-01-04", "ProductA", 100, 12)
]

schema = StructType([
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("quantity", IntegerType(), True)
])

df = spark.createDataFrame(data, schema)
df.createOrReplaceTempView("sales")
df.show()

+----------+--------+-----+--------+
|      date| product|price|quantity|
+----------+--------+-----+--------+
|2025-01-01|ProductA|  120|      10|
|2025-01-02|ProductB|  200|       5|
|2025-01-02|ProductA|  150|       8|
|2025-01-03|ProductC|  300|       3|
|2025-01-03|ProductB|  250|       6|
|2025-01-04|ProductA|  100|      12|
+----------+--------+-----+--------+



#### 3.2. Data Transformation (Python API) ####

Apply filters, aggregations, and new columns.

In [4]:
revenue_df = df.withColumn("revenue", col("price") * col("quantity")) \
               .groupby("product") \
               .agg(sum("revenue").alias("total_revenue"))

revenue_df.show()

+--------+-------------+
| product|total_revenue|
+--------+-------------+
|ProductA|         3600|
|ProductB|         2500|
|ProductC|          900|
+--------+-------------+



#### 3.3. SQL Queries ####

You can also run SQL directly against the temporary view.

In [5]:
result = spark.sql("""
    SELECT product,
           SUM(price * quantity) AS total_revenue,
           COUNT(*) AS order_count
    FROM sales
    GROUP BY product
    ORDER BY total_revenue DESC
""")

result.show()

+--------+-------------+-----------+
| product|total_revenue|order_count|
+--------+-------------+-----------+
|ProductA|         3600|          3|
|ProductB|         2500|          2|
|ProductC|          900|          1|
+--------+-------------+-----------+



#### 3.4. Reading / Writing Data ####

In real projects you read from files, tables, or external sources. Example: read a CSV.

In [6]:
# Write the DataFrame to a temporary CSV.
# NOTE: On Windows, Spark's native CSV writer requires the Hadoop native library
# (winutils.exe/hadoop.dll) which is not bundled with pyspark alone.
# For small tutorial data, we write via pandas, then read back with Spark.
import shutil
csv_path = "sales_csv"
shutil.rmtree(csv_path, ignore_errors=True)  # remove leftover Spark output dir if present

pdf = df.toPandas()
pdf.to_csv(csv_path, index=False)
print(f"Wrote CSV to: {csv_path}")

# Read it back with Spark
df_read = spark.read.option("header", "true").csv(csv_path)
df_read.show()


c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Wrote CSV to: sales_csv
+----------+--------+-----+--------+
|      date| product|price|quantity|
+----------+--------+-----+--------+
|2025-01-01|ProductA|  120|      10|
|2025-01-02|ProductB|  200|       5|
|2025-01-02|ProductA|  150|       8|
|2025-01-03|ProductC|  300|       3|
|2025-01-03|ProductB|  250|       6|
|2025-01-04|ProductA|  100|      12|
+----------+--------+-----+--------+



#### 4. Pandas API on Spark ####

If you are familiar with `pandas`, you can scale your workloads by using the Pandas API on Spark. It provides a pandas-like API that executes distributedly.

In [7]:
# Convert Spark DataFrame to pandas-on-Spark DataFrame
ps_df = ps.DataFrame(df)

# Perform pandas-like operations
ps_revenue = ps_df.assign(revenue = ps_df["price"] * ps_df["quantity"]) \
                  .groupby("product")["revenue"].sum() \
                  .reset_index()
print(ps_revenue)

# You can switch back to Spark Dataframe easily
spark_revenue = ps_revenue.to_spark()
spark_revenue.show()


c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


    product  revenue
0  ProductA     3600
1  ProductB     2500
2  ProductC      900


c:\Users\Admin\Documents\Duke's Workspace\Programming\Data Science\Data Techniques\Data Engineering\spark_fundamentals\.venv\Lib\site-packages\pyspark\pandas\utils.py:1053: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+--------+-------+
| product|revenue|
+--------+-------+
|ProductA|   3600|
|ProductB|   2500|
|ProductC|    900|
+--------+-------+



**Key benefit:** Write code that works with small data (Pandas) and large data (Spark) without changing syntax.

### 5. Structured Streaming ###

Structured Streaming processes real-time data using the same DataFrame / SQL API. In this example, we simulate a stream from a rate generator and compute a sliding window average of the value.

In [8]:
# Create a streaming DataFrame that reads from a rate source (1 row per second)
stream_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

# Add a timestamp and a simple transformation
stream_df = stream_df.withColumn("value", col("value") * 10) # scale value

# Compute a 10-second tumbling window average of the value
windowed_stream = stream_df \
    .withWatermark("timestamp", "10 seconds") \
    .groupBy(window("timestamp", "10 seconds")) \
    .agg(avg("value").alias("avg_value"))

# Write the output to the console (for demonstration)
# checkpointLocation is required for streaming queries on Windows
checkpoint_dir = os.path.join(os.getcwd(), "stream_checkpoint")
if os.path.exists(checkpoint_dir):
    shutil.rmtree(checkpoint_dir)

query = windowed_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("checkpointLocation", checkpoint_dir) \
    .trigger(processingTime = "5 seconds") \
    .start()

# Let the stream run for 20 seconds, then stop
time.sleep(20)
query.stop()
print("Streaming query stopped.")


Streaming query stopped.


**Note:** In production, you would write to a sink like Kafka, Parquet, or a Delta table.

### 6. Machine Learning (MLlib) ###

MLlib provides scalable machine learning algorithms. While rarely used in production (many companies prefer external libraries), it’s still valuable for feature engineering and model training on huge datasets.

Here is a simple pipeline that trains a Linear Regression on synthetic data.

In [9]:
# Create synthetic data: two numeric feature columns and a label
ml_data = spark.range(0, 1000) \
    .withColumn("feature1", randn(seed=42)) \
    .withColumn("feature2", randn(seed=123)) \
    .withColumn("label", col("feature1") * 2 + col("feature2") * 0.5 + 1)

# Assemble the two numeric columns into a single vector column
assembler = VectorAssembler(inputCols=["feature1", "feature2"], outputCol="features_vec")
lr = LinearRegression(featuresCol="features_vec", labelCol="label")

pipeline = Pipeline(stages=[assembler, lr])

# Split data
train, test = ml_data.randomSplit([0.8, 0.2], seed=42)

# Train model
model = pipeline.fit(train)

# Generate predictions
predictions = model.transform(test)

# Evaluate using RegressionEvaluator on the predictions DataFrame
evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print("RMSE:", rmse)


RMSE: 5.943334914149856e-16


### 7. Spark Core and RDDs ###

RDDs (Resilient Distributed Datasets) are the low‑level API of Spark. They are not recommended for most use cases because DataFrames provide better optimization and ease of use. However, it’s good to know them for debugging or custom algorithms.

In [10]:
# Create an RDD from a Python list
data_rdd = spark.sparkContext.parallelize([1, 2, 3, 4, 5])

# Apply transformations (map, filter, reduce)
squared_rdd = data_rdd.map(lambda x: x * x)
filtered_rdd = squared_rdd.filter(lambda x: x > 5)
result = filtered_rdd.collect()  # collect action

print("RDD result:", result)

# Convert RDD to DataFrame (but prefer creating DataFrames directly)
df_from_rdd = filtered_rdd.map(lambda x: (x,)).toDF(["squared_value"])
df_from_rdd.show()

RDD result: [9, 16, 25]
+-------------+
|squared_value|
+-------------+
|            9|
|           16|
|           25|
+-------------+



### 8. Closing the Session ###

In [11]:
spark.stop()